# Current FDTD API: a pulse, a material film and a named record

Use the public `wavebit.fdtd` module with shared `Material` and explicit `Units`. This notebook runs on NumPy/CPU. Geometry and wavelength are in micrometres; time is seconds and pulse bandwidth is Hz. The source electric amplitude is V/m. Electric records are V/m and magnetic records are A/m.

This small periodic-x/y domain records Ey below a dielectric film. PML is added outside the physical z domain. The example demonstrates execution and linear amplitude scaling, **not** a calibrated spectrum or spatial/time refinement. See the [current FDTD guide](../../docs/getting-started/fdtd.md).

In [ ]:
from wavebit import Material, Units, fdtd

problem = fdtd.Problem(
    units=Units(length="um"),
    scene=fdtd.Scene(
        domain=fdtd.Domain(size=(.1, .1, 1.6)),
        boundaries=fdtd.Boundaries(x=fdtd.Periodic(), y=fdtd.Periodic(), z=fdtd.Open()),
        structures=(fdtd.Structure("film", shape=fdtd.SnappedSlab(z=(-.1, .1)),
                                   material=Material(epsilon=2.25)),),
    ),
    sources=(fdtd.PlaneWave("illumination", plane=fdtd.Plane(z=.5),
        pulse=fdtd.GaussianPulse(wavelength=.8, frequency_sigma_hz=2e13)),),
    outputs=fdtd.Outputs(monitors=(
        fdtd.TimeSeriesMonitor("probe", at=(0., 0., -.3), component="Ey"),)),
)

## Separate physical input from numerical approximation

The problem owns the domain, material film, pulse source and requested monitor. The grid, duration, CFL and absorber belong to `Numerics`. A `TimeSeriesMonitor` samples the nearest Yee point for its component; it does not interpolate to an arbitrary point.

In [ ]:
numerics = fdtd.Numerics(grid=fdtd.UniformGrid(spacing=.025),
    duration=fdtd.Duration(periods=20, source="illumination"), absorber=fdtd.PML(cells=12))
report = fdtd.check(problem, numerics=numerics, backend="numpy")
assert report.ok
prepared = fdtd.prepare(problem, numerics=numerics, backend="numpy")
result = prepared.run(problem)
print("samples:", len(result["probe"].times_seconds), "time unit: s; units:", result["probe"].units)

## Reuse the prepared calculation with a new source amplitude

The second problem keeps the same static layout and changes a live amplitude. Each run receives the complete problem. In this linear material, halving the source halves Ey. The earlier record remains unchanged.

In [ ]:
import numpy as np

original_values = np.asarray(result["probe"].values).copy()
half_source = problem.sources[0].replace(amplitude=.5)
half_problem = problem.replace(sources=(half_source,))
half_result = prepared.run(half_problem)
np.testing.assert_allclose(half_result["probe"].values, .5 * original_values)
np.testing.assert_array_equal(result["probe"].values, original_values)
print("Source amplitude halved; the original record is unchanged.")

Continue with the [general FDTD guide](../../docs/guide/fdtd/general-fdtd.md), [reuse contract](../../docs/guide/common/lifecycle.md) and [refinement learning path](../../docs/learn/fdtd/index.md). The advanced periodic FDTD notebook in this folder explicitly uses the compatibility interface.